# Release16 Trend BC Slice Failure Drilldown v1

Purpose: turn the release16 trend-target BC official replay failure into an explicit model-selection handoff before any new training run. This notebook does not promote or retain an actor. It checks whether the failure is a same-family BC weight-sweep problem or a target/objective problem that must be repaired first.

Governance:
- Canonical baseline: `soxlbacktest_qlib_walkforward_baseline_v1` and the current retention frontier gate.
- Direct comparison: `release16_wf2020_trend_target_bc_official_replay_v1`.
- Role: diagnose / route next model-selection objective.
- Changed layer: `model_selection` handoff only; no runtime promotion and no broker behavior.
- Target windows: `wf_2020`, `wf_2022`, `wf_2023_2024`.
- Acceptance criteria: MaxDD `< 20%`, walk-forward relative return `>= 30pp`, full slice/drift gate, and SOXLBacktest baseline comparison preserved.
- External reference: repo-owned SOXLBacktest qlib walk-forward baseline and release16 official replay sidecars.


In [1]:
from __future__ import annotations

import json
import math
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "AGENTS.md").exists():
    ROOT = Path("/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

EXPERIMENT = "release16_trend_bc_slice_failure_drilldown_v1"
SOURCE_EXPERIMENT = "release16_wf2020_trend_target_bc_official_replay_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
SOURCE_CARD = ROOT / "runtime" / "model_selection" / "card" / SOURCE_EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 20.0
REQUIRED_WINDOWS = ["wf_2020", "wf_2022", "wf_2023_2024"]

required_files = [
    "summary.json",
    "bc_candidate_metrics.csv",
    "bc_slice_metrics.csv",
    "bc_action_distribution.csv",
    "bc_fit_metrics.csv",
    "bc_drift_scores.csv",
    "target_coverage.csv",
    "drawdown_interval_diagnostic.csv",
    "soxlbacktest_baseline_comparison.csv",
    "candidate_gate_summary.csv",
    "failure_attribution.csv",
]
missing = [name for name in required_files if not (SOURCE_CARD / name).exists()]
if missing:
    raise FileNotFoundError(f"missing source sidecars: {missing}")

source_summary = json.loads((SOURCE_CARD / "summary.json").read_text(encoding="utf-8"))
frames = {
    name[:-4]: pd.read_csv(SOURCE_CARD / name)
    for name in required_files
    if name.endswith(".csv")
}

source_readout = pd.DataFrame(
    [
        {
            "source_experiment": SOURCE_EXPERIMENT,
            "decision": source_summary.get("decision"),
            "primary_blocker": source_summary.get("primary_blocker"),
            "next_mechanism": source_summary.get("next_mechanism"),
            "selected_candidate": source_summary.get("selected_candidate"),
            "rows": sum(len(frame) for frame in frames.values()),
        }
    ]
)
display(source_readout)


,source_experiment,decision,primary_blocker,next_mechanism,selected_candidate,rows
0,release16_wf2020_trend_target_bc_official_repl...,release16_wf2020_trend_target_bc_slice_failed_...,bc_candidate_or_slice_gate_failed,repair_trend_slice_sampling_or_state_loss_befo...,None,336


In [2]:
candidate = frames["bc_candidate_metrics"].copy()
boolean_cols = [
    "relative_gate",
    "max_dd_gate_strict",
    "train_slice_gate",
    "holdout_slice_gate",
    "drift_gate",
    "candidate_gate_strict",
]
for col in boolean_cols:
    candidate[col] = candidate[col].astype(bool)

selected_by_fold = source_summary.get("selected_by_fold", {}) or {}
selected_lookup = {
    str(fold): str(payload.get("selected_candidate"))
    for fold, payload in selected_by_fold.items()
    if isinstance(payload, dict)
}

candidate["target_relative_return_pp"] = TARGET_RELATIVE_RETURN_PP
candidate["target_max_dd_pct"] = TARGET_MAX_DD_PCT
candidate["relative_gap_to_target_pp"] = (TARGET_RELATIVE_RETURN_PP - candidate["relative_return"]).clip(lower=0.0)
candidate["dd_gap_to_target_pct"] = (candidate["max_dd"] - TARGET_MAX_DD_PCT).clip(lower=0.0)
candidate["selected_in_official_replay"] = candidate.apply(
    lambda row: selected_lookup.get(str(row["fold"])) == str(row["candidate"]), axis=1
)


def candidate_failure_mode(row: pd.Series) -> str:
    blockers: list[str] = []
    if bool(row["candidate_gate_strict"]):
        return "pass"
    if not bool(row["relative_gate"]):
        blockers.append("relative_return")
    if not bool(row["max_dd_gate_strict"]):
        blockers.append("drawdown")
    if not bool(row["train_slice_gate"]):
        blockers.append("train_slice")
    if not bool(row["holdout_slice_gate"]):
        blockers.append("holdout_slice")
    if not bool(row["drift_gate"]):
        blockers.append("drift")
    return "+".join(blockers) if blockers else "unknown_gate_failure"

candidate["failure_mode"] = candidate.apply(candidate_failure_mode, axis=1)

candidate_failure_matrix = candidate[
    [
        "fold",
        "candidate",
        "selected_in_official_replay",
        "relative_return",
        "max_dd",
        "avg_pos",
        "trade_count",
        "relative_gap_to_target_pp",
        "dd_gap_to_target_pct",
        "relative_delta_vs_runnable",
        "max_dd_delta_vs_runnable",
        "relative_gate",
        "max_dd_gate_strict",
        "train_slice_gate",
        "holdout_slice_gate",
        "drift_gate",
        "candidate_gate_strict",
        "failure_mode",
    ]
].sort_values(["fold", "selected_in_official_replay", "candidate"], ascending=[True, False, True])

candidate_failure_matrix.to_csv(CARD / "candidate_failure_matrix.csv", index=False)
display(candidate_failure_matrix)


,fold,candidate,selected_in_official_replay,relative_return,max_dd,avg_pos,trade_count,relative_gap_to_target_pp,dd_gap_to_target_pct,relative_delta_vs_runnable,max_dd_delta_vs_runnable,relative_gate,max_dd_gate_strict,train_slice_gate,holdout_slice_gate,drift_gate,candidate_gate_strict,failure_mode
0,wf_2020,risk_trend_balanced,True,12.286463,26.568904,0.436140,43,17.713537,6.568904,-9.995176,5.462748,False,False,True,True,False,False,relative_return+drawdown+drift
2,wf_2020,class_balanced_trend,False,-27.132620,22.795332,0.325694,39,57.132620,2.795332,-49.414259,1.689176,False,False,True,True,False,False,relative_return+drawdown+drift
3,wf_2020,trend_balanced,False,-14.305880,28.878449,0.379579,38,44.305880,8.878449,-36.587519,7.772293,False,False,True,True,False,False,relative_return+drawdown+drift
1,wf_2020,trend_heavy,False,-18.215406,25.653898,0.358140,37,48.215406,5.653898,-40.497045,4.547742,False,False,True,True,False,False,relative_return+drawdown+drift
4,wf_2022,risk_trend_balanced,True,65.269018,24.711475,0.144049,34,0.000000,4.711475,-7.181842,9.638836,True,False,True,False,False,False,drawdown+holdout_slice+drift
5,wf_2022,class_balanced_trend,False,62.473442,28.076103,0.154252,36,0.000000,8.076103,-9.977418,13.003464,True,False,True,False,False,False,drawdown+holdout_slice+drift
6,wf_2022,trend_balanced,False,51.702750,35.841360,0.149368,34,0.000000,15.841360,-20.748110,20.768721,True,False,True,False,False,False,drawdown+holdout_slice+drift
7,wf_2022,trend_heavy,False,51.283359,36.255466,0.149701,34,0.000000,16.255466,-21.167501,21.182827,True,False,True,False,False,False,drawdown+holdout_slice+drift
8,wf_2023_2024,risk_trend_balanced,True,19.035720,14.803893,0.204210,34,10.964280,0.000000,4.733658,0.844459,False,True,False,False,True,False,relative_return+train_slice+holdout_slice
9,wf_2023_2024,class_balanced_trend,False,33.651053,20.974794,0.267497,39,0.000000,0.974794,19.348991,7.015360,True,False,False,False,False,False,drawdown+train_slice+holdout_slice+drift


In [3]:
gate = frames["candidate_gate_summary"].copy()
fail = frames["failure_attribution"].copy()
fold_gap_summary = gate.merge(fail, on="fold", how="left", suffixes=("", "_selected"))
fold_gap_summary["selected_relative_gap_to_target_pp"] = (
    TARGET_RELATIVE_RETURN_PP - fold_gap_summary["selected_relative_return"]
).clip(lower=0.0)
fold_gap_summary["selected_dd_gap_to_target_pct"] = (
    fold_gap_summary["selected_max_dd"] - TARGET_MAX_DD_PCT
).clip(lower=0.0)
fold_gap_summary["full_gate_dead_end"] = fold_gap_summary["full_gate_candidates"].eq(0)
fold_gap_summary["no_relative_solution"] = fold_gap_summary["relative_gate_candidates"].eq(0)
fold_gap_summary["no_strict_dd_solution"] = fold_gap_summary["strict_dd_gate_candidates"].eq(0)
fold_gap_summary["no_holdout_slice_solution"] = fold_gap_summary["holdout_slice_gate_candidates"].eq(0)
fold_gap_summary["same_bc_weight_sweep_viable"] = fold_gap_summary["full_gate_candidates"].gt(0)


def fold_blocker(row: pd.Series) -> str:
    if bool(row["same_bc_weight_sweep_viable"]):
        return "same_bc_weight_sweep_has_candidate"
    blockers: list[str] = []
    if bool(row["no_relative_solution"]):
        blockers.append("no_relative_solution")
    if bool(row["no_strict_dd_solution"]):
        blockers.append("no_strict_dd_solution")
    if bool(row["no_holdout_slice_solution"]):
        blockers.append("no_holdout_slice_solution")
    if int(row.get("candidate_pool_drift_gate_count", 0)) == 0:
        blockers.append("no_drift_gate_solution")
    return "+".join(blockers) if blockers else "joint_gate_failure"

fold_gap_summary["dominant_blocker"] = fold_gap_summary.apply(fold_blocker, axis=1)
fold_gap_summary = fold_gap_summary[
    [
        "fold",
        "selected_candidate",
        "selected_relative_return",
        "selected_max_dd",
        "selected_avg_pos",
        "selected_relative_gap_to_target_pp",
        "selected_dd_gap_to_target_pct",
        "candidate_rows",
        "full_gate_candidates",
        "relative_gate_candidates",
        "strict_dd_gate_candidates",
        "holdout_slice_gate_candidates",
        "drift_gate_candidates",
        "same_bc_weight_sweep_viable",
        "failure_profile",
        "dominant_blocker",
        "recommended_next_mechanism",
    ]
].sort_values("fold")

fold_gap_summary.to_csv(CARD / "fold_gap_summary.csv", index=False)
display(fold_gap_summary)


,fold,selected_candidate,selected_relative_return,selected_max_dd,selected_avg_pos,selected_relative_gap_to_target_pp,selected_dd_gap_to_target_pct,candidate_rows,full_gate_candidates,relative_gate_candidates,strict_dd_gate_candidates,holdout_slice_gate_candidates,drift_gate_candidates,same_bc_weight_sweep_viable,failure_profile,dominant_blocker,recommended_next_mechanism
0,wf_2020,risk_trend_balanced,12.286463,26.568904,0.436140,17.713537,6.568904,4,0,0,0,4,0,False,joint_return_drawdown+drift_vs_runnable_reference,no_relative_solution+no_strict_dd_solution+no_...,drawdown_drift_penalty_or_checkpoint_scoring
1,wf_2022,risk_trend_balanced,65.269018,24.711475,0.144049,0.000000,4.711475,4,0,4,0,0,0,False,sparse_slice_fit+drawdown_control_gap+drift_vs...,no_strict_dd_solution+no_holdout_slice_solutio...,risk_state_or_drawdown_drift_checkpoint_objective
2,wf_2023_2024,risk_trend_balanced,19.035720,14.803893,0.204210,10.964280,0.000000,4,0,1,1,0,1,False,sparse_slice_fit+underparticipation_relative_gap,no_holdout_slice_solution,risk_state_or_drawdown_drift_checkpoint_objective


In [4]:
def load_slice_detail_rows(source: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for record in source.to_dict("records"):
        for split, column in [
            ("train", "train_slice_details_json"),
            ("holdout", "holdout_slice_details_json"),
        ]:
            payload = record.get(column)
            if not isinstance(payload, str) or not payload.strip():
                continue
            details = json.loads(payload)
            for slice_name, detail in details.items():
                if not isinstance(detail, dict):
                    continue
                required = bool(detail.get("required", False))
                gate_passed = bool(detail.get("gate", True))
                fail_reasons: list[str] = []
                if required and not bool(detail.get("sample_gate", True)):
                    fail_reasons.append("sample")
                if required and not bool(detail.get("accuracy_gate", True)):
                    fail_reasons.append("accuracy")
                if required and not bool(detail.get("diversity_gate", True)):
                    fail_reasons.append("diversity")
                if required and not gate_passed and not fail_reasons:
                    fail_reasons.append("gate")
                rows.append(
                    {
                        "fold": record["fold"],
                        "candidate": record["candidate"],
                        "bc_split": split,
                        "slice": slice_name,
                        "required": required,
                        "gate": gate_passed,
                        "failed_required_gate": bool(required and not gate_passed),
                        "fail_reasons": "+".join(fail_reasons),
                        "samples": detail.get("samples", 0),
                        "full_target_count": detail.get("full_target_count", 0),
                        "accuracy": detail.get("accuracy"),
                        "min_accuracy": detail.get("min_accuracy"),
                        "label_unique_actions": detail.get("label_unique_actions"),
                        "predicted_unique_actions": detail.get("predicted_unique_actions"),
                        "min_predicted_unique": detail.get("min_predicted_unique"),
                        "predicted_dominant_share": np.nan,
                        "skip_reason": detail.get("skip_reason", ""),
                    }
                )
    return pd.DataFrame(rows)

slice_detail = load_slice_detail_rows(candidate)
raw_slice_metrics = frames["bc_slice_metrics"].copy()
slice_detail = slice_detail.merge(
    raw_slice_metrics[
        [
            "fold",
            "candidate",
            "bc_split",
            "slice",
            "predicted_dominant_share",
            "label_dominant_share",
        ]
    ],
    on=["fold", "candidate", "bc_split", "slice"],
    how="left",
    suffixes=("", "_raw"),
)
slice_detail["predicted_dominant_share"] = slice_detail["predicted_dominant_share_raw"].combine_first(
    slice_detail["predicted_dominant_share"]
)
slice_detail = slice_detail.drop(columns=["predicted_dominant_share_raw"])
slice_detail.to_csv(CARD / "slice_gate_detail.csv", index=False)

slice_failure_summary = (
    slice_detail.groupby(["fold", "bc_split", "slice"], as_index=False)
    .agg(
        candidates=("candidate", "nunique"),
        required_candidates=("required", "sum"),
        failed_required_candidates=("failed_required_gate", "sum"),
        min_samples=("samples", "min"),
        median_accuracy=("accuracy", "median"),
        min_predicted_unique_actions=("predicted_unique_actions", "min"),
        max_predicted_dominant_share=("predicted_dominant_share", "max"),
        fail_reasons=("fail_reasons", lambda values: ",".join(sorted({str(v) for v in values if str(v)}))),
    )
    .sort_values(["fold", "bc_split", "failed_required_candidates", "slice"], ascending=[True, True, False, True])
)
slice_failure_summary.to_csv(CARD / "slice_failure_summary.csv", index=False)

failed_required = slice_failure_summary[slice_failure_summary["failed_required_candidates"].gt(0)]
display(failed_required if not failed_required.empty else slice_failure_summary.head(12))


,fold,bc_split,slice,candidates,required_candidates,failed_required_candidates,min_samples,median_accuracy,min_predicted_unique_actions,max_predicted_dominant_share,fail_reasons
18,wf_2022,holdout,drawdown_control_state,4,4,4,6,0.250000,2.0,0.666667,accuracy
20,wf_2022,holdout,is_downtrend_state,4,4,4,24,0.458333,4.0,0.416667,accuracy
17,wf_2022,holdout,aux_state_head_release_state,4,4,1,35,0.685714,4.0,0.542857,accuracy
21,wf_2022,holdout,is_rebound_cluster,4,4,1,37,0.662162,4.0,0.540541,accuracy
34,wf_2023_2024,holdout,drawdown_control_state,4,4,4,7,0.357143,2.0,0.571429,accuracy
36,wf_2023_2024,holdout,is_downtrend_state,4,4,4,97,0.536082,3.0,0.474227,accuracy
39,wf_2023_2024,holdout,trend_participation_target_state,4,4,1,29,0.603448,1.0,1.000000,diversity
42,wf_2023_2024,train,drawdown_control_state,4,4,4,28,0.535714,3.0,0.607143,accuracy
47,wf_2023_2024,train,trend_participation_target_state,4,4,1,109,0.770642,1.0,1.000000,diversity


In [5]:
action_dist = frames["bc_action_distribution"].copy()
target_action_distribution = action_dist.pivot_table(
    index=["fold"], columns="action", values="share", aggfunc="first", fill_value=0.0
).reset_index()
for col in [c for c in target_action_distribution.columns if c != "fold"]:
    target_action_distribution.rename(columns={col: f"target_action_{col}_share"}, inplace=True)

position = candidate.groupby("fold", as_index=False).agg(
    candidate_count=("candidate", "nunique"),
    min_avg_pos=("avg_pos", "min"),
    max_avg_pos=("avg_pos", "max"),
    min_trade_count=("trade_count", "min"),
    max_trade_count=("trade_count", "max"),
)
selected_position = candidate[candidate["selected_in_official_replay"]][
    ["fold", "candidate", "avg_pos", "trade_count", "relative_return", "max_dd"]
].rename(
    columns={
        "candidate": "selected_candidate",
        "avg_pos": "selected_avg_pos",
        "trade_count": "selected_trade_count",
        "relative_return": "selected_relative_return",
        "max_dd": "selected_max_dd",
    }
)

drawdown = frames["drawdown_interval_diagnostic"].copy()
drawdown_summary = drawdown.groupby("fold", as_index=False).agg(
    worst_interval_dd=("max_dd", "max"),
    max_interval_avg_pos=("interval_avg_pos", "max"),
    max_interval_trades=("interval_trades", "max"),
    longest_interval_days=("interval_days", "max"),
)

action_position_summary = (
    position.merge(selected_position, on="fold", how="left")
    .merge(drawdown_summary, on="fold", how="left")
    .merge(target_action_distribution, on="fold", how="left")
    .sort_values("fold")
)
action_position_summary.to_csv(CARD / "action_position_summary.csv", index=False)
display(action_position_summary)


,fold,candidate_count,min_avg_pos,max_avg_pos,min_trade_count,max_trade_count,selected_candidate,selected_avg_pos,selected_trade_count,selected_relative_return,selected_max_dd,worst_interval_dd,max_interval_avg_pos,max_interval_trades,longest_interval_days,target_action_0_share,target_action_1_share,target_action_2_share,target_action_3_share,target_action_4_share
0,wf_2020,4,0.325694,0.436140,37,43,risk_trend_balanced,0.436140,43,12.286463,26.568904,28.878449,0.370043,6,41,0.124130,0.027262,0.382251,0.015081,0.451276
1,wf_2022,4,0.144049,0.154252,34,36,risk_trend_balanced,0.144049,34,65.269018,24.711475,36.255466,0.149622,34,249,0.226131,0.231156,0.170854,0.010050,0.361809
2,wf_2023_2024,4,0.204210,0.323783,34,41,risk_trend_balanced,0.204210,34,19.035720,14.803893,35.312162,0.225183,19,115,0.256667,0.256667,0.115000,0.005000,0.366667


In [6]:
baseline = frames["soxlbacktest_baseline_comparison"].copy()
selected_candidates = candidate[candidate["selected_in_official_replay"]][["fold", "candidate"]]
baseline_gap_summary = baseline.merge(selected_candidates, on=["fold", "candidate"], how="inner")
baseline_gap_summary = baseline_gap_summary[
    [
        "fold",
        "candidate",
        "baseline",
        "candidate_relative_return",
        "baseline_relative_return",
        "relative_delta",
        "candidate_max_dd",
        "baseline_max_dd",
        "max_dd_delta",
        "candidate_avg_pos",
        "baseline_avg_pos",
    ]
].sort_values(["fold", "baseline"])
baseline_gap_summary.to_csv(CARD / "baseline_gap_summary.csv", index=False)
display(baseline_gap_summary)


,fold,candidate,baseline,candidate_relative_return,baseline_relative_return,relative_delta,candidate_max_dd,baseline_max_dd,max_dd_delta,candidate_avg_pos,baseline_avg_pos
0,wf_2020,risk_trend_balanced,runnable_reference,12.286463,22.281639,-9.995176,26.568904,21.106156,5.462748,0.436140,0.425202
1,wf_2020,risk_trend_balanced,soxl_legacy_default,12.286463,161.001897,-148.715434,26.568904,54.275144,-27.706240,0.436140,0.791023
2,wf_2020,risk_trend_balanced,soxl_risk_floor,12.286463,-30.618705,42.905168,26.568904,12.785207,13.783697,0.436140,0.171011
3,wf_2022,risk_trend_balanced,runnable_reference,65.269018,72.450860,-7.181842,24.711475,15.072639,9.638836,0.144049,0.087837
4,wf_2022,risk_trend_balanced,soxl_legacy_default,65.269018,7.191701,58.077317,24.711475,84.918053,-60.206578,0.144049,0.866930
5,wf_2022,risk_trend_balanced,soxl_risk_floor,65.269018,70.543521,-5.274504,24.711475,18.514585,6.196890,0.144049,0.102318
6,wf_2023_2024,risk_trend_balanced,runnable_reference,19.035720,14.302062,4.733658,14.803893,13.959434,0.844459,0.204210,0.197766
7,wf_2023_2024,risk_trend_balanced,soxl_legacy_default,19.035720,5.538546,13.497174,14.803893,32.461810,-17.657917,0.204210,0.310435
8,wf_2023_2024,risk_trend_balanced,soxl_risk_floor,19.035720,7.109208,11.926512,14.803893,1.061804,13.742089,0.204210,0.013221


In [7]:
same_bc_weight_sweep_allowed = bool(fold_gap_summary["same_bc_weight_sweep_viable"].any())
full_gate_folds = int(fold_gap_summary["full_gate_candidates"].gt(0).sum())
strict_dd_solution_folds = int(fold_gap_summary["strict_dd_gate_candidates"].gt(0).sum())
relative_solution_folds = int(fold_gap_summary["relative_gate_candidates"].gt(0).sum())
holdout_slice_solution_folds = int(fold_gap_summary["holdout_slice_gate_candidates"].gt(0).sum())

selected_min_relative = float(fold_gap_summary["selected_relative_return"].min())
selected_worst_dd = float(fold_gap_summary["selected_max_dd"].max())
max_relative_gap = float(fold_gap_summary["selected_relative_gap_to_target_pp"].max())
max_dd_gap = float(fold_gap_summary["selected_dd_gap_to_target_pct"].max())

recommended_next_mechanism = "risk_state_drawdown_drift_checkpoint_objective"
next_action = (
    "train risk-state/drawdown-drift checkpoint objective before another official replay; "
    "do not extend the same release16 trend BC weight sweep"
)

artifact_paths = {
    "candidate_failure_matrix": str((CARD / "candidate_failure_matrix.csv").relative_to(ROOT)),
    "fold_gap_summary": str((CARD / "fold_gap_summary.csv").relative_to(ROOT)),
    "slice_gate_detail": str((CARD / "slice_gate_detail.csv").relative_to(ROOT)),
    "slice_failure_summary": str((CARD / "slice_failure_summary.csv").relative_to(ROOT)),
    "action_position_summary": str((CARD / "action_position_summary.csv").relative_to(ROOT)),
    "baseline_gap_summary": str((CARD / "baseline_gap_summary.csv").relative_to(ROOT)),
}

recommendation = {
    "experiment": EXPERIMENT,
    "source_experiment": SOURCE_EXPERIMENT,
    "decision": "release16_trend_bc_same_sweep_stopped_objective_handoff",
    "primary_blocker": "target_revision_ready",
    "recommended_next_mechanism": recommended_next_mechanism,
    "next_action": next_action,
    "same_bc_weight_sweep_allowed": same_bc_weight_sweep_allowed,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "ppo_allowed_next": False,
    "bc_screen_allowed_next": False,
    "next_training_allowed_after_target_revision": True,
    "full_gate_folds": full_gate_folds,
    "strict_dd_solution_folds": strict_dd_solution_folds,
    "relative_solution_folds": relative_solution_folds,
    "holdout_slice_solution_folds": holdout_slice_solution_folds,
    "selected_min_relative_return_pp": selected_min_relative,
    "selected_worst_max_drawdown_pct": selected_worst_dd,
    "max_relative_gap_to_target_pp": max_relative_gap,
    "max_drawdown_gap_to_target_pct": max_dd_gap,
    "target_relative_return_pp": TARGET_RELATIVE_RETURN_PP,
    "target_max_dd_pct": TARGET_MAX_DD_PCT,
    "required_wf_windows": REQUIRED_WINDOWS,
    "artifact_paths": artifact_paths,
}

selected_by_fold_summary = {
    row["fold"]: {
        "selected_candidate": row["selected_candidate"],
        "selected_relative_return": float(row["selected_relative_return"]),
        "selected_max_dd": float(row["selected_max_dd"]),
        "selected_avg_pos": float(row["selected_avg_pos"]),
        "failure_profile": row["failure_profile"],
        "dominant_blocker": row["dominant_blocker"],
    }
    for row in fold_gap_summary.to_dict("records")
}

summary = {
    **recommendation,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "source_path": str(SOURCE_CARD.relative_to(ROOT)),
    "reference_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
    "direct_comparison_baseline": SOURCE_EXPERIMENT,
    "next_mechanism": recommended_next_mechanism,
    "selected_by_fold": selected_by_fold_summary,
    "min_wf_relative_return_pp": selected_min_relative,
    "worst_wf_max_drawdown_pct": selected_worst_dd,
    "training_handoff": [
        {
            "decision": "ready",
            "action": recommended_next_mechanism,
            "next_action": next_action,
            "source_experiment": SOURCE_EXPERIMENT,
        }
    ],
    "risk_notes": [
        "No release16 trend BC candidate passed the full gate in any required walk-forward fold.",
        "wf_2020 has no relative-return or strict-DD candidate; wf_2022 has no strict-DD or holdout-slice candidate.",
        "wf_2023_2024 has isolated DD/drift pass but still fails relative-return or slice gates depending on candidate.",
        "Proceeding to PPO or actor retention before repairing the objective would preserve the same failure mode.",
    ],
}

(CARD / "recommendation.json").write_text(json.dumps(recommendation, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")

conclusion_lines = [
    "# Release16 Trend BC Slice Failure Drilldown v1",
    "",
    "Decision: `release16_trend_bc_same_sweep_stopped_objective_handoff`.",
    "",
    f"- Same-family BC weight sweep allowed: `{same_bc_weight_sweep_allowed}`.",
    f"- Full-gate folds: `{full_gate_folds}/{len(REQUIRED_WINDOWS)}`.",
    f"- Strict-DD solution folds: `{strict_dd_solution_folds}/{len(REQUIRED_WINDOWS)}`.",
    f"- Relative-return solution folds: `{relative_solution_folds}/{len(REQUIRED_WINDOWS)}`.",
    f"- Holdout-slice solution folds: `{holdout_slice_solution_folds}/{len(REQUIRED_WINDOWS)}`.",
    f"- Selected min WF relative return: `{selected_min_relative:.2f}` pp versus target `{TARGET_RELATIVE_RETURN_PP:.2f}` pp.",
    f"- Selected worst WF MaxDD: `{selected_worst_dd:.2f}%` versus target `< {TARGET_MAX_DD_PCT:.2f}%`.",
    f"- Next mechanism: `{recommended_next_mechanism}`.",
    "",
    "Conclusion: do not start another release16 trend-target BC weight sweep. The next runnable model work should train a risk-state / drawdown-drift checkpoint objective and keep SOXLBacktest qlib walk-forward as the baseline comparison.",
]
(CARD / "conclusion.md").write_text("\n".join(conclusion_lines) + "\n", encoding="utf-8")

display(Markdown("\n".join(conclusion_lines)))


# Release16 Trend BC Slice Failure Drilldown v1

Decision: `release16_trend_bc_same_sweep_stopped_objective_handoff`.

- Same-family BC weight sweep allowed: `False`.
- Full-gate folds: `0/3`.
- Strict-DD solution folds: `1/3`.
- Relative-return solution folds: `2/3`.
- Holdout-slice solution folds: `1/3`.
- Selected min WF relative return: `12.29` pp versus target `30.00` pp.
- Selected worst WF MaxDD: `26.57%` versus target `< 20.00%`.
- Next mechanism: `risk_state_drawdown_drift_checkpoint_objective`.

Conclusion: do not start another release16 trend-target BC weight sweep. The next runnable model work should train a risk-state / drawdown-drift checkpoint objective and keep SOXLBacktest qlib walk-forward as the baseline comparison.